<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Randomise

**[Randomise](https://learning.nextia-ai.com/courses/experiments/m02/randomise/)** · Experiments and Causal Impact: Did Our Change Work? · Module 2, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see why a coin flip removes confounders: the groups it makes are alike before the change, in every column, even in the ones you did not measure.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Confounders, from [Causes and comparisons](https://learning.nextia-ai.com/courses/experiments/m01/causes-and-comparisons/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m02/randomise/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M02-L01-randomise/randomise-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "logs/customers.csv.gz": "0e9bd54ac28ef98bf94f6d1d0abe1bf0f8e51f67384d7b08c94b0b9310ab320a",
    "logs/truth.json": "8e2cd0eda8b31c1fba87687dfaf14b598d24921cf38734e8d18a21d8126b009f"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The users were different before the block

The logs show the customers in the four weeks after the recommendation block went live. `pre_spend` is what each customer spent in the 8 weeks **before** the launch: the block cannot have changed it.

> **Predict.** Did the customers who later used the block spend more before it existed?

In [ ]:
logs = pd.read_csv("data/logs/customers.csv.gz")
cols = ["pre_spend", "pre_orders", "tenure_days", "new_customer"]
logs.groupby("used_block")[cols].mean()

> **Check.** You should see higher values for the users (`used_block` 1) in the first three columns: pre_spend 51.77 against 41.28 dollars.

The groups differed before the change. So "users spent more" mixes two things: what the block did, and who the users were.

## 2. One number for "how different": the standardised difference

To compare columns with different units, divide the difference of the means by the typical spread: (mean₁ − mean₀) ÷ √((sd₁² + sd₀²) ÷ 2). Below 0.1 is usually called balanced.

In [ ]:
def std_diff(df, group, col):
    a, b = df.loc[df[group] == 1, col], df.loc[df[group] == 0, col]
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)

for c in cols:
    print(f"{c:<13} {std_diff(logs, 'used_block', c):+.3f}")

> **Check.** You should see `pre_spend +0.076` and `tenure_days +0.143`.

The users were with Larkfield longer, and fewer of them were new. By the "below 0.1" rule, pre_spend looks balanced; section 4 shows that its gap is still far larger than chance.

## 3. Now flip a coin

Suppose we had shown the block by a coin flip instead. The next cell gives each customer a random group with NumPy, then compares the groups on the same columns.

> **Predict.** Will the coin's groups differ in pre_spend?

In [ ]:
rng = np.random.default_rng(1)
logs["coin"] = (rng.random(len(logs)) < 0.5).astype(int)
for c in cols:
    print(f"{c:<13} {std_diff(logs, 'coin', c):+.3f}")

> **Check.** You should see four values close to 0: pre_spend +0.001.

The coin knows nothing about the customers, so it cannot pick the better ones. That is why a coin flip removes confounders: also the ones that are not in the data. (The logs hide one on purpose: whether the customer has a garden. The coin balances it too.)

## 4. How close is "close"? Flip it 1,000 times

One flip gives a small difference by chance. The next cell flips the coin 1,000 times and draws the differences of pre_spend.

In [ ]:
import matplotlib.pyplot as plt

x = logs["pre_spend"].to_numpy()
diffs = []
for i in range(1000):
    g = np.random.default_rng(100 + i).random(len(x)) < 0.5
    diffs.append(x[g].mean() - x[~g].mean())
diffs = np.array(diffs)
users_gap = logs.loc[logs.used_block == 1, "pre_spend"].mean() - logs.loc[logs.used_block == 0, "pre_spend"].mean()
plt.hist(diffs, bins=40, color="#9aa7c7")
plt.axvline(users_gap, color="crimson", label="users − non-users")
plt.xlabel("difference in pre_spend (dollars)")
plt.legend()
plt.show()
print(f"coin flips: 95% of differences between {np.percentile(diffs, 2.5):+.2f} and {np.percentile(diffs, 97.5):+.2f}; "
      f"users − non-users: {users_gap:+.2f}")

> **Check.** You should see `coin flips: 95% of differences between -2.52 and +2.76; users − non-users: +10.49`.

The coin's differences are small and centred on 0. The users' difference is far outside them: it is not chance, it is self-selection.

## 5. Your turn: a coin flip with seed 2

> **Your turn.** Make a new coin with `np.random.default_rng(2)` (one `random()` number per customer, below 0.5 is group 1), store it in a column `coin2`, and compute the standardised difference of `visits` between its groups with `std_diff`. Round it to 3 decimals and put it in `sd_visits`.

In [ ]:
logs["coin2"] = (np.random.default_rng(2).random(len(logs)) < 0.5).astype(int)
sd_visits = round(std_diff(logs, "coin2", "visits"), 3)
print(sd_visits)

In [ ]:
expect('sd_visits', sd_visits, 0.004)

## Next

A coin flip makes comparable groups. Which unit we flip for, a customer or a session, is the next question of the [lesson page](https://learning.nextia-ai.com/courses/experiments/m02/randomise/), which also has the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Choose the metrics](https://learning.nextia-ai.com/courses/experiments/m02/choose-the-metrics/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m02/randomise/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.